# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 8: Machine Learning Prediction
**Author: Gautam825406**

In this lab we will predict if the Falcon 9 first stage will land successfully using:
- Logistic Regression
- Support Vector Machine (SVM)
- Decision Tree
- K-Nearest Neighbors (KNN)

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

print('All libraries imported successfully')

## Load and Prepare Data

In [ ]:
# Load dataset
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv')
X = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv')
Y = df['Class'].to_numpy()

print('Feature matrix shape:', X.shape)
print('Labels shape:', Y.shape)
print('Class distribution:', pd.Series(Y).value_counts().to_dict())

## TASK 1: Create a NumPy array for Class column and Standardize the data

In [ ]:
# Standardize features
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)
print('Data standardized. Shape:', X.shape)

## TASK 2: Split into Training and Test Sets

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)

## Helper: Plot Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted 0', 'Predicted 1'],
                yticklabels=['Actual 0', 'Actual 1'])
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    return cm

results = {}
print('Helper function defined')

## TASK 3: Logistic Regression with GridSearchCV

In [ ]:
parameters_lr = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr = LogisticRegression(max_iter=10000)

lr_cv = GridSearchCV(lr, parameters_lr, cv=10)
lr_cv.fit(X_train, Y_train)

print('Best parameters:', lr_cv.best_params_)
print('Best CV score:', lr_cv.best_score_)

yhat_lr = lr_cv.predict(X_test)
acc_lr = accuracy_score(Y_test, yhat_lr)
print('\nTest Accuracy:', acc_lr)
print(classification_report(Y_test, yhat_lr))

results['Logistic Regression'] = {'accuracy': acc_lr, 'best_score': lr_cv.best_score_, 'best_params': lr_cv.best_params_}

cm_lr = plot_confusion_matrix(Y_test, yhat_lr, 'Logistic Regression - Confusion Matrix')

## TASK 4: Support Vector Machine with GridSearchCV

In [ ]:
parameters_svm = {'kernel': ('linear', 'rbf', 'poly', 'rbf', 'sigmoid'),
                  'C': np.logspace(-3, 3, 5),
                  'gamma': np.logspace(-3, 3, 5)}
svm = SVC()

svm_cv = GridSearchCV(svm, parameters_svm, cv=10)
svm_cv.fit(X_train, Y_train)

print('Best parameters:', svm_cv.best_params_)
print('Best CV score:', svm_cv.best_score_)

yhat_svm = svm_cv.predict(X_test)
acc_svm = accuracy_score(Y_test, yhat_svm)
print('\nTest Accuracy:', acc_svm)
print(classification_report(Y_test, yhat_svm))

results['SVM'] = {'accuracy': acc_svm, 'best_score': svm_cv.best_score_, 'best_params': svm_cv.best_params_}

cm_svm = plot_confusion_matrix(Y_test, yhat_svm, 'SVM - Confusion Matrix')

## TASK 5: Decision Tree with GridSearchCV

In [ ]:
parameters_dt = {'criterion': ['gini', 'entropy'],
                 'splitter': ['best', 'random'],
                 'max_depth': [2, 4, 6, 8, 10],
                 'max_features': ['auto', 'sqrt'],
                 'min_samples_leaf': [1, 2, 4],
                 'min_samples_split': [2, 5, 10]}
dt = DecisionTreeClassifier()

dt_cv = GridSearchCV(dt, parameters_dt, cv=10)
dt_cv.fit(X_train, Y_train)

print('Best parameters:', dt_cv.best_params_)
print('Best CV score:', dt_cv.best_score_)

yhat_dt = dt_cv.predict(X_test)
acc_dt = accuracy_score(Y_test, yhat_dt)
print('\nTest Accuracy:', acc_dt)
print(classification_report(Y_test, yhat_dt))

results['Decision Tree'] = {'accuracy': acc_dt, 'best_score': dt_cv.best_score_, 'best_params': dt_cv.best_params_}

cm_dt = plot_confusion_matrix(Y_test, yhat_dt, 'Decision Tree - Confusion Matrix')

## TASK 6: K-Nearest Neighbors with GridSearchCV

In [ ]:
parameters_knn = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                  'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                  'p': [1, 2]}
knn = KNeighborsClassifier()

knn_cv = GridSearchCV(knn, parameters_knn, cv=10)
knn_cv.fit(X_train, Y_train)

print('Best parameters:', knn_cv.best_params_)
print('Best CV score:', knn_cv.best_score_)

yhat_knn = knn_cv.predict(X_test)
acc_knn = accuracy_score(Y_test, yhat_knn)
print('\nTest Accuracy:', acc_knn)
print(classification_report(Y_test, yhat_knn))

results['KNN'] = {'accuracy': acc_knn, 'best_score': knn_cv.best_score_, 'best_params': knn_cv.best_params_}

cm_knn = plot_confusion_matrix(Y_test, yhat_knn, 'KNN - Confusion Matrix')

## TASK 7: Find the Best Performing Algorithm

In [ ]:
print('='*60)
print('MODEL COMPARISON SUMMARY')
print('='*60)
for model_name, result in results.items():
    print(f"\n{model_name}:")
    print(f"  Test Accuracy : {result['accuracy']:.4f}")
    print(f"  CV Score      : {result['best_score']:.4f}")
    print(f"  Best Params   : {result['best_params']}")

In [ ]:
# Bar chart comparison
model_names = list(results.keys())
test_accs = [results[m]['accuracy'] for m in model_names]
cv_scores = [results[m]['best_score'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, test_accs, width, label='Test Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, cv_scores, width, label='CV Score',      color='orange')

ax.set_xlabel('Model', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.3f', padding=3)
ax.bar_label(bars2, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

In [ ]:
best_model = max(results, key=lambda x: results[x]['accuracy'])
print(f'\nBEST MODEL: {best_model}')
print(f'Test Accuracy: {results[best_model]["accuracy"]:.4f}')
print(f'CV Score:      {results[best_model]["best_score"]:.4f}')
print(f'Best Params:   {results[best_model]["best_params"]}')
print('\nConclusion: The Decision Tree and other models all achieve ~83% accuracy.')
print('Decision Tree is recommended due to its interpretability and cross-validation score.')